# Data Exploration

> These are Go notebooks: In order to use the GoNB Jupyter Kernel, please install GoNB from here: https://github.com/janpfeifer/gonb

Note also that for local package development, you can put: `!*go mod edit -replace "github.com/umbralcalc/antimicrobial-resistance=/path/to/antimicrobial-resistance"` at the top of any cell.
\
Visualise the Fingertips AMR surveillance data downloaded via `./dat/fetch_fingertips.sh`.

In [ ]:
import (
	"encoding/csv"
	"os"
	"strconv"

	"github.com/go-echarts/go-echarts/v2/charts"
	"github.com/go-echarts/go-echarts/v2/opts"
	"github.com/go-gota/gota/dataframe"
	"github.com/go-gota/gota/series"
	gonb_echarts "github.com/janpfeifer/gonb-echarts"
	"github.com/umbralcalc/stochadex/pkg/analysis"
	"gonum.org/v1/gonum/floats"
)

// loadCSV reads a Fingertips CSV and returns records (excluding header).
func loadCSV(path string) [][]string {
	f, err := os.Open(path)
	if err != nil {
		panic(err)
	}
	defer f.Close()
	r := csv.NewReader(f)
	r.LazyQuotes = true
	all, err := r.ReadAll()
	if err != nil {
		panic(err)
	}
	return all // [0] is header, [1:] are records
}

// quarterToFloat converts "2019 Q1" to 2019.0, "2019 Q2" to 2019.25, etc.
func quarterToFloat(q string) float64 {
	if len(q) < 7 {
		return 0
	}
	year, _ := strconv.ParseFloat(q[:4], 64)
	qn, _ := strconv.ParseFloat(string(q[6]), 64)
	return year + (qn-1)*0.25
}

%%

fmt.Println("Loaded utilities")

## England-level: broad-spectrum prescribing % vs E. coli 3GC resistance %

The Fingertips `broadspectrum_pct` (indicator 92167) and `ecoli_cephalosporin_resistance_pct` (indicator 92559) are both quarterly at ICB sub-location level. We filter for the England-level aggregate.

In [ ]:
import (
	"strconv"

	"github.com/go-echarts/go-echarts/v2/charts"
	"github.com/go-echarts/go-echarts/v2/opts"
	"github.com/go-gota/gota/dataframe"
	"github.com/go-gota/gota/series"
	gonb_echarts "github.com/janpfeifer/gonb-echarts"
	"github.com/umbralcalc/stochadex/pkg/analysis"
	"gonum.org/v1/gonum/floats"
)

%%

// Load England-level prescribing and resistance time series
prescRows := loadCSV("../dat/fingertips_broadspectrum_pct.csv")
resistRows := loadCSV("../dat/fingertips_ecoli_cephalosporin_resistance_pct.csv")

// Header index: 6=Area Type, 11=Time period, 12=Value
prescTimes := make([]float64, 0)
prescValues := make([]float64, 0)
for _, row := range prescRows[1:] {
	if row[6] == "England" {
		t := quarterToFloat(row[11])
		v, err := strconv.ParseFloat(row[12], 64)
		if err == nil && t > 0 {
			prescTimes = append(prescTimes, t)
			prescValues = append(prescValues, v)
		}
	}
}

resistTimes := make([]float64, 0)
resistValues := make([]float64, 0)
for _, row := range resistRows[1:] {
	if row[6] == "England" {
		t := quarterToFloat(row[11])
		v, err := strconv.ParseFloat(row[12], 64)
		if err == nil && t > 0 {
			resistTimes = append(resistTimes, t)
			resistValues = append(resistValues, v)
		}
	}
}

// Build dataframes for plotting
dfPresc := dataframe.New(
	series.New(prescTimes, series.Float, "QUARTER"),
	series.New(prescValues, series.Float, "BROADSPECTRUM_PCT"),
)
dfResist := dataframe.New(
	series.New(resistTimes, series.Float, "QUARTER"),
	series.New(resistValues, series.Float, "RESISTANCE_PCT"),
)

fmt.Printf("Prescribing: %d quarters, Resistance: %d quarters\n",
	dfPresc.Nrow(), dfResist.Nrow())

// Plot broad-spectrum prescribing % over time
line := analysis.NewLinePlotFromDataFrame(&dfPresc, "QUARTER", "BROADSPECTRUM_PCT")
line.SetGlobalOptions(
	charts.WithTitleOpts(opts.Title{
		Title:  "England: broad-spectrum prescribing % (cephalosporin/quinolone/co-amoxiclav)",
		Bottom: "1%",
	}),
	charts.WithYAxisOpts(opts.YAxis{
		Name: "%",
		Min:  floats.Min(prescValues) - 0.5,
		Max:  floats.Max(prescValues) + 0.5,
	}),
	charts.WithXAxisOpts(opts.XAxis{
		Name: "Quarter (year)",
		Min:  floats.Min(prescTimes) - 0.25,
		Max:  floats.Max(prescTimes) + 0.25,
	}),
)
gonb_echarts.Display(line, "width: 1024px; height: 400px; background: white;")

// Plot E. coli 3GC resistance % over time
lineR := analysis.NewLinePlotFromDataFrame(&dfResist, "QUARTER", "RESISTANCE_PCT")
lineR.SetGlobalOptions(
	charts.WithTitleOpts(opts.Title{
		Title:  "England: E. coli 3rd-gen cephalosporin resistance %",
		Bottom: "1%",
	}),
	charts.WithYAxisOpts(opts.YAxis{
		Name: "%",
		Min:  floats.Min(resistValues) - 1.0,
		Max:  floats.Max(resistValues) + 1.0,
	}),
	charts.WithXAxisOpts(opts.XAxis{
		Name: "Quarter (year)",
		Min:  floats.Min(resistTimes) - 0.25,
		Max:  floats.Max(resistTimes) + 0.25,
	}),
)
gonb_echarts.Display(lineR, "width: 1024px; height: 400px; background: white;")

## ICB sub-location cross-sectional scatter

For the most recent common quarter, plot each ICB sub-location's broad-spectrum prescribing % against its E. coli 3GC resistance %.

In [ ]:
import (
	"encoding/csv"
	"os"
	"strconv"

	"github.com/go-echarts/go-echarts/v2/charts"
	"github.com/go-echarts/go-echarts/v2/opts"
	"github.com/go-gota/gota/dataframe"
	"github.com/go-gota/gota/series"
	gonb_echarts "github.com/janpfeifer/gonb-echarts"
	"github.com/umbralcalc/stochadex/pkg/analysis"
	"gonum.org/v1/gonum/floats"
)

%%

// Build ICB-level cross-sectional data for the latest common quarter
// Header: 4=Area Code, 5=Area Name, 6=Area Type, 11=Time period, 12=Value
prescRows := loadCSV("../dat/fingertips_broadspectrum_pct.csv")
resistRows := loadCSV("../dat/fingertips_ecoli_cephalosporin_resistance_pct.csv")

// Collect ICB sub-location records
prescICB := make(map[string]map[string]float64) // code -> period -> value
resistICB := make(map[string]map[string]float64)
for _, row := range prescRows[1:] {
	if row[6] == "ICB sub-locations" {
		v, err := strconv.ParseFloat(row[12], 64)
		if err == nil {
			if prescICB[row[4]] == nil {
				prescICB[row[4]] = make(map[string]float64)
			}
			prescICB[row[4]][row[11]] = v
		}
	}
}
for _, row := range resistRows[1:] {
	if row[6] == "ICB sub-locations" {
		v, err := strconv.ParseFloat(row[12], 64)
		if err == nil {
			if resistICB[row[4]] == nil {
				resistICB[row[4]] = make(map[string]float64)
			}
			resistICB[row[4]][row[11]] = v
		}
	}
}

// Find latest common quarter
commonPeriods := make(map[string]bool)
for _, periods := range prescICB {
	for p := range periods {
		commonPeriods[p] = true
	}
}
latestQ := ""
for _, periods := range resistICB {
	for p := range periods {
		if commonPeriods[p] && p > latestQ {
			latestQ = p
		}
	}
}

// Build scatter data
xs := make([]float64, 0)
ys := make([]float64, 0)
for code, periods := range prescICB {
	pv, pOk := periods[latestQ]
	rv, rOk := resistICB[code][latestQ]
	if pOk && rOk {
		xs = append(xs, pv)
		ys = append(ys, rv)
	}
}

dfScatter := dataframe.New(
	series.New(xs, series.Float, "BROADSPECTRUM_PCT"),
	series.New(ys, series.Float, "RESISTANCE_PCT"),
)

fmt.Printf("Cross-sectional scatter: %d ICB sub-locations for %s\n", len(xs), latestQ)

scatter := analysis.NewScatterPlotFromDataFrame(
	&dfScatter, "BROADSPECTRUM_PCT", "RESISTANCE_PCT",
)
scatter.SetGlobalOptions(
	charts.WithTitleOpts(opts.Title{
		Title:  fmt.Sprintf("ICB sub-locations: prescribing vs resistance (%s)", latestQ),
		Bottom: "1%",
	}),
	charts.WithXAxisOpts(opts.XAxis{
		Name: "Broad-spectrum prescribing (%)",
		Min:  floats.Min(xs) - 1,
		Max:  floats.Max(xs) + 1,
	}),
	charts.WithYAxisOpts(opts.YAxis{
		Name: "E. coli 3GC resistance (%)",
		Min:  floats.Min(ys) - 2,
		Max:  floats.Max(ys) + 2,
	}),
)
gonb_echarts.Display(scatter, "width: 1024px; height: 500px; background: white;")

## E. coli bacteraemia rates by acute trust

12-month rolling E. coli BSI rates for selected acute trusts with the most data coverage.

In [ ]:
import (
	"strconv"

	"github.com/go-echarts/go-echarts/v2/charts"
	"github.com/go-echarts/go-echarts/v2/opts"
	"github.com/go-gota/gota/dataframe"
	"github.com/go-gota/gota/series"
	gonb_echarts "github.com/janpfeifer/gonb-echarts"
	"github.com/umbralcalc/stochadex/pkg/analysis"
	"gonum.org/v1/gonum/floats"
)

%%

// Load E. coli bacteraemia annual data and plot for selected trusts
bactRows := loadCSV("../dat/fingertips_ecoli_bacteraemia_annual.csv")

// Header: 5=Area Name, 6=Area Type, 11=Time period, 12=Value
// Collect trust-level data, pick trusts with >=10 data points
trustData := make(map[string][][2]float64) // name -> [(year, value), ...]
for _, row := range bactRows[1:] {
	if row[6] != "Acute Trust" {
		continue
	}
	v, err := strconv.ParseFloat(row[12], 64)
	if err != nil {
		continue
	}
	// Time period is like "2019/20" — extract first year
	year, err := strconv.ParseFloat(row[11][:4], 64)
	if err != nil {
		continue
	}
	trustData[row[5]] = append(trustData[row[5]], [2]float64{year, v})
}

// Select top 8 trusts by number of data points
type trustCount struct {
	name  string
	count int
}
counts := make([]trustCount, 0)
for name, data := range trustData {
	counts = append(counts, trustCount{name, len(data)})
}
// Simple sort
for i := range counts {
	for j := i + 1; j < len(counts); j++ {
		if counts[j].count > counts[i].count {
			counts[i], counts[j] = counts[j], counts[i]
		}
	}
}

allTimes := make([]float64, 0)
allValues := make([]float64, 0)
allNames := make([]string, 0)
for i := 0; i < 8 && i < len(counts); i++ {
	name := counts[i].name
	for _, d := range trustData[name] {
		allTimes = append(allTimes, d[0])
		allValues = append(allValues, d[1])
		allNames = append(allNames, name)
	}
}

dfTrust := dataframe.New(
	series.New(allTimes, series.Float, "YEAR"),
	series.New(allValues, series.Float, "RATE"),
	series.New(allNames, series.String, "TRUST"),
)

trustLine := analysis.NewLinePlotFromDataFrame(&dfTrust, "YEAR", "RATE", "TRUST")
trustLine.SetGlobalOptions(
	charts.WithTitleOpts(opts.Title{
		Title:  "E. coli bacteraemia rates by acute trust (per 100k)",
		Bottom: "1%",
	}),
	charts.WithYAxisOpts(opts.YAxis{Name: "Rate per 100k"}),
	charts.WithXAxisOpts(opts.XAxis{
		Name: "Financial year",
		Min:  floats.Min(allTimes) - 0.5,
		Max:  floats.Max(allTimes) + 0.5,
	}),
	charts.WithLegendOpts(opts.Legend{
		Show:   opts.Bool(true),
		Bottom: "5%",
		Type:   "scroll",
	}),
)
gonb_echarts.Display(trustLine, "width: 1024px; height: 500px; background: white;")